# 매출 YoY vs 수출 HS코드 YoY 상관관계 분석 (읽기 전용)

DB에 어떤 것도 쓰지 않습니다 (SELECT만 사용).

## v2 변경 사항

### A. 수입 버전과 공통

1. **52/53주 회계 캘린더 대응 (핵심 버그 수정)**
   AAPL/AMAT 같은 기업은 분기말이 매년 며칠씩 이동하다가 가끔 다음 달 1~2일로 넘어간다
   (예: AAPL FY2023 Q2 = 2023-04-01, Q3 = 2023-07-01).
   기존 `_month_end()` 는 무조건 앞으로 밀어 그 달 말일로 만들었기 때문에 이런 날짜가
   한 달 통째로 밀려나 직전 분기와의 간격이 120일이 되었고, `tol_days=100` 연속성 검사에서
   시계열이 끊겨 해당 티커가 통째로 탈락했다.
   → `_month_end_fiscal()` (가장 가까운 월말로 스냅) 로 교체.

   **주의**: 무역 월별 데이터에는 기존 동작(`_month_end_calendar`)을 그대로 써야 한다.

2. **티커별 cycle_group 단일화** — 한 티커가 두 그룹에 걸치면 시계열이 쪼개져 YoY가 망가진다.
3. **중복 INSERT 대비 `ORDER BY ... id`** — `drop_duplicates(keep="last")` 가 항상 확정치를 남기도록.
4. **표본 수 필터 강화** — `n_periods >= 4` → `MIN_CORR_PERIODS = 12`, p-value 컬럼 추가.
5. **조회 함수 시그니처 통일** (`by_abs`, `min_periods`, `min_abs_corr is not None`)

### B. 수출 버전 전용

6. **수출 최신버전 self-join 수정 (중요)**
   기존 쿼리는 `GROUP BY hs_code` 로 `MAX(created_at)` 을 잡았다. 이러면 테이블이 월별로
   나눠 INSERT되어 각 월이 서로 다른 `created_at` 을 갖는 경우, HS코드당 **가장 최근에
   INSERT된 한 달치 행만** 남고 나머지 전 기간이 사라진다.

   ```
   원본 6개월  ->  기존 join 통과 1행 (2020-06 만)
               ->  수정 join 통과 6행 (전체)
   ```

   → `GROUP BY hs_code, date_month_end` 로 변경. 한 배치에 같은 `created_at` 이 찍히는
   경우에도 결과가 동일하므로 어느 쪽이든 안전하다.
   join 전후 행 수를 출력하니 실행 후 반드시 확인할 것.

7. **무역 패널 중복 방지**
   `pivot_table(aggfunc="sum")` 은 같은 (hs_code, date) 행이 둘 이상이면 값을 합산해
   조용히 2배로 부풀린다. 명시적 중복 제거 단계를 추가하고 제거 건수를 출력한다.

8. **매출 데이터 부족 시 조기 경고**
   기존 코드는 `revenue_filtered` 가 비면 `.dt` 접근에서
   `AttributeError: Can only use .dt accessor with datetimelike values` 로 터졌다.
   원인(DB에 분기 수 부족)을 알려주는 메시지로 교체.

## 처리 순서

### ① 매출 데이터
1. `US_IS_from_FMP`에서 티커별 분기 매출 로드
2. 최근 시점부터 거꾸로 봐서 **연속 20분기 이상** 인 티커만 채택
3. 분기말 월 기준 3그룹 분리, 티커별 최빈 그룹으로 단일화
4. 그룹별로 티커마다 YoY 매출성장률 계산

### ② 수출 HS코드 데이터
1. 월별 실측 데이터 로드 (HS코드×월 단위 최신 버전만), **연속 60개월 이상**인 HS코드만 채택
2. `rolling(3, min_periods=3).sum()` → 3가지 분기 패턴 전부의 분기합계를 한 번에 커버
3. `.pct_change(12)` → HS코드별 분기 YoY

### ③ 상관계수 계산
각 티커가 실제로 리포트한 날짜 그대로 HS YoY 패널에서 조회 후 `corrwith()` 벡터화 계산.


In [1]:

# -*- coding: utf-8 -*-
from __future__ import annotations
import os, sys
from pathlib import Path


def _find_project_root(module_name="DATA", max_up=6):
    here = Path.cwd()
    for base in [here, *list(here.parents)[:max_up]]:
        if (base / module_name).is_dir():
            return base
    return None


try:
    from DATA.stock_invest_function import *
except ModuleNotFoundError:
    _root = _find_project_root("DATA")
    if _root is None:
        raise ModuleNotFoundError("DATA 패키지를 찾을 수 없습니다. 프로젝트 루트에서 실행해 주세요.")
    sys.path.insert(0, str(_root))
    from DATA.stock_invest_function import *
    print(f"[경로 자동 보정] DATA 모듈을 {_root} 에서 찾아 sys.path에 추가했습니다.")

import numpy as np
import pandas as pd
from sqlalchemy import create_engine, text
import warnings
warnings.filterwarnings("ignore")

db_info = {"host": get_db_host(), "port": 3307, "user": "stox7412",
           "password": "Apt106503!~", "database": "investar"}


def make_engine(db_info):
    url = (f"mysql+pymysql://{db_info['user']}:{db_info['password']}"
           f"@{db_info['host']}:{int(db_info['port'])}/{db_info['database']}?charset=utf8mb4")
    return create_engine(url, pool_pre_ping=True, pool_recycle=1800,
                          connect_args={"connect_timeout": 10, "read_timeout": 60, "write_timeout": 60})


engine = make_engine(db_info)
TABLE_REVENUE = "US_IS_from_FMP"
TABLE_EXPORT_MONTHLY = "us_trade_export_monthly_with_forecast"
TABLE_IMPORT_MONTHLY = "us_trade_import_monthly_with_forecast"
DIRECTION = "export"   # 이 노트북 전용: "export" 또는 "import"

# 이 노트북은 SELECT 만 사용합니다. INSERT/UPDATE/DELETE/ALTER 없음.


In [2]:

# ============================================================
# 공통 헬퍼
# ============================================================

def _month_end_calendar(ts):
    """달력 기준 월말으로 스냅. **무역 월별 데이터 전용.**
        2023-07-01 -> 2023-07-31
        2023-07-31 -> 2023-07-31
    """
    return pd.Timestamp(ts) + pd.offsets.MonthEnd(0)


def _month_end_fiscal(ts):
    """가장 가까운 월말로 스냅. **기업 분기말 전용.**

    52/53주 회계 캘린더(AAPL, AMAT 등)에서 분기말이 다음 달 1~2일로 넘어가는 경우를
    원래 속한 달로 되돌린다.
        2023-07-01 -> 2023-06-30
        2023-09-30 -> 2023-09-30
        2021-05-02 -> 2021-04-30
        2021-01-31 -> 2021-01-31

    이 함수를 쓰지 않으면 2022-12-31 -> 2023-04-30 처럼 간격이 120일로 벌어져
    get_trailing_consecutive_run(tol_days=100) 이 시계열을 끊어버린다.
    """
    return (pd.Timestamp(ts) - pd.Timedelta(days=15)) + pd.offsets.MonthEnd(0)


def get_trailing_consecutive_run(dates, tol_days=100):
    """'최근 시점부터 거꾸로' 봤을 때, 간격이 tol_days 이내로 이어지는 연속 구간만 추출."""
    dates = sorted(pd.to_datetime(d) for d in dates)
    if not dates:
        return []
    run = [dates[-1]]
    for i in range(len(dates) - 2, -1, -1):
        if (run[-1] - dates[i]).days <= tol_days:
            run.append(dates[i])
        else:
            break
    return sorted(run)


def compute_revenue_yoy(series: pd.Series, tol_days: int = 45) -> pd.Series:
    """기업 매출 시계열의 YoY. 1년 전 가장 가까운(±tol_days) 리포트와 비교."""
    series = series.sort_index()
    idx = series.index
    target_dates = idx - pd.DateOffset(years=1)
    base_vals = series.reindex(target_dates, method="nearest", tolerance=pd.Timedelta(days=tol_days))
    base_vals.index = idx
    with np.errstate(invalid="ignore", divide="ignore"):
        yoy = (series.values / base_vals.values - 1.0) * 100.0
    return pd.Series(yoy, index=idx).replace([np.inf, -np.inf], np.nan).dropna()


def load_trade_monthly_long(engine, direction="import", verbose=True):
    """무역 월별 데이터 로드. 날짜는 _month_end_calendar 로 스냅 (fiscal 아님!).

    export 분기 수정 사항
    ---------------------
    최신 버전만 남기는 self-join의 기준을 `hs_code` 단독 -> `hs_code + date_month_end` 로 변경.
    기존 방식은 테이블이 월별로 나눠 INSERT되어 각 월의 created_at이 다를 경우
    HS코드당 마지막 한 달치만 남기고 전 기간을 날려버린다.
    """
    if direction == "import":
        sql = text(f"""
            SELECT hs_code_6d AS hs_code, date, impDlr AS value
            FROM {TABLE_IMPORT_MONTHLY}
            WHERE forecast_flag = 0 AND impDlr IS NOT NULL
        """)
    elif direction == "export":
        with engine.connect() as conn:
            cols = [r[0] for r in conn.execute(text(f"SHOW COLUMNS FROM {TABLE_EXPORT_MONTHLY}")).fetchall()]
        vcol = "created_at" if "created_at" in cols else ("input_date" if "input_date" in cols else None)

        # join 없이 전체를 먼저 세어서 self-join이 몇 행을 걸러내는지 확인
        if verbose:
            with engine.connect() as conn:
                n_raw = conn.execute(text(f"""
                    SELECT COUNT(*) FROM {TABLE_EXPORT_MONTHLY}
                    WHERE is_forecast = 0 AND expDlr IS NOT NULL
                """)).scalar()
            print(f"[export] self-join 적용 전 행 수: {n_raw:,}")

        # ★ 수정: GROUP BY 에 date_month_end 포함 (기존에는 hs_code 뿐이었음)
        vcol_join = f"""
            JOIN (SELECT hs_code, date_month_end, MAX({vcol}) AS mx
                  FROM {TABLE_EXPORT_MONTHLY}
                  WHERE is_forecast = 0 AND expDlr IS NOT NULL
                  GROUP BY hs_code, date_month_end) l
              ON t.hs_code = l.hs_code
             AND t.date_month_end = l.date_month_end
             AND t.{vcol} = l.mx
        """ if vcol else ""
        if verbose and vcol is None:
            print("[export] 버전 컬럼(created_at/input_date) 없음 — self-join 생략")

        sql = text(f"""
            SELECT t.hs_code AS hs_code, t.date_month_end AS date, t.expDlr AS value
            FROM {TABLE_EXPORT_MONTHLY} t
            {vcol_join}
            WHERE t.is_forecast = 0 AND t.expDlr IS NOT NULL
        """)
    else:
        raise ValueError("direction must be 'import' or 'export'")

    with engine.connect() as conn:
        rows = conn.execute(sql).fetchall()
    df = pd.DataFrame(rows, columns=["hs_code", "date", "value"])
    df["date"] = pd.to_datetime(df["date"]).apply(_month_end_calendar)
    df["hs_code"] = df["hs_code"].astype(str).str.zfill(6)

    if verbose:
        print(f"[{direction}] 로드 완료: {len(df):,}행 / HS코드 {df['hs_code'].nunique():,}개")

    # pivot_table(aggfunc="sum") 이 중복을 조용히 합산하는 것을 방지
    n_dup = df.duplicated(subset=["hs_code", "date"]).sum()
    if n_dup:
        df = df.drop_duplicates(subset=["hs_code", "date"], keep="last")
        if verbose:
            print(f"[{direction}] (hs_code, date) 중복 {n_dup:,}행 제거")
    return df


# ------------------------------------------------------------
# 조회 함수 (두 함수 시그니처 통일)
#   by_abs=False : 양의 상관 높은 순 (기본)
#   by_abs=True  : 절댓값 큰 순 (음의 상관도 함께 보고 싶을 때)
# ------------------------------------------------------------

def _top_by_corr(df, n, min_abs_corr, by_abs, min_periods=None):
    df = df.copy()
    df["correlation"] = pd.to_numeric(df["correlation"], errors="coerce")
    df = df.dropna(subset=["correlation"])

    if min_abs_corr is not None:                    # 0 도 유효한 값이므로 is not None
        df = df[df["correlation"].abs() >= min_abs_corr]
    if min_periods is not None and "n_periods" in df.columns:
        df = df[df["n_periods"] >= min_periods]

    key = df["correlation"].abs() if by_abs else df["correlation"]
    return (df.assign(_key=key)
              .sort_values("_key", ascending=False, kind="mergesort")   # 동점 시 원본 순서 유지
              .drop(columns="_key")
              .head(n)
              .reset_index(drop=True))


def get_top_hscodes_for_ticker(correlation_df, ticker, n=10,
                               min_abs_corr=None, by_abs=False, min_periods=None):
    return _top_by_corr(correlation_df[correlation_df["ticker"] == ticker],
                        n, min_abs_corr, by_abs, min_periods)


def get_top_tickers_for_hscode(correlation_df, hs_code, n=10,
                               min_abs_corr=None, by_abs=False, min_periods=None):
    hs_code = str(hs_code).zfill(6)
    return _top_by_corr(correlation_df[correlation_df["hs_code"] == hs_code],
                        n, min_abs_corr, by_abs, min_periods)


## ① 매출 로드 + 연속 20분기 필터 + 분기주기 3그룹 분리 + YoY

In [3]:

MIN_QUARTERS = 20
QUARTER_TOL_DAYS = 100
FORCE_SINGLE_CYCLE_GROUP = True   # 티커별 cycle_group을 최빈값 하나로 강제 통일

# 중복 INSERT 대비: id(AUTO_INCREMENT PK)가 있으면 정렬에 포함시켜
# drop_duplicates(keep="last") 가 항상 '가장 나중에 저장된 행'(=확정치)을 남기게 한다.
with engine.connect() as conn:
    _rev_cols = {r[0] for r in conn.execute(text(f"SHOW COLUMNS FROM {TABLE_REVENUE}")).fetchall()}
_order_by = "ticker, date, id" if "id" in _rev_cols else "ticker, date"

sql = text(f"""
    SELECT ticker, date, value AS revenue
    FROM {TABLE_REVENUE}
    WHERE item = 'revenue' AND period IN ('Q1','Q2','Q3','Q4')
      AND value IS NOT NULL AND value > 0
    ORDER BY {_order_by}
""")
with engine.connect() as conn:
    rows = conn.execute(sql).fetchall()
revenue_raw = pd.DataFrame(rows, columns=["ticker", "date", "revenue"])

# ★ 기업 분기말이므로 fiscal 버전 사용 (calendar 버전 쓰면 AAPL/AMAT 등이 탈락한다)
revenue_raw["date"] = pd.to_datetime(revenue_raw["date"]).apply(_month_end_fiscal)

_n_before = len(revenue_raw)
revenue_raw = revenue_raw.drop_duplicates(subset=["ticker", "date"], keep="last")
_per_ticker = len(revenue_raw) / max(revenue_raw["ticker"].nunique(), 1)
print(f"[매출] 원본: 티커 {revenue_raw['ticker'].nunique():,}개 / 레코드 {len(revenue_raw):,}건 "
      f"(티커당 평균 {_per_ticker:.1f}분기, 중복 제거 {_n_before - len(revenue_raw):,}건)")
if _per_ticker < MIN_QUARTERS:
    print(f"  ⚠️  티커당 평균 분기 수가 {MIN_QUARTERS}보다 적습니다. "
          f"US_FMP_FS_1_RUN_UPDATE.py --quarters 60 으로 과거 데이터를 채우세요.")

kept_rows, n_qualified = [], 0
for ticker, g in revenue_raw.groupby("ticker"):
    run_dates = get_trailing_consecutive_run(g["date"].tolist(), tol_days=QUARTER_TOL_DAYS)
    if len(run_dates) >= MIN_QUARTERS:
        n_qualified += 1
        kept_rows.append(g[g["date"].isin(run_dates)])
print(f"[매출] 연속 {MIN_QUARTERS}분기 이상 만족 티커: {n_qualified:,}개")

# 빈 결과일 때 .dt AttributeError 대신 원인을 알려주고 중단
if not kept_rows:
    raise RuntimeError(
        f"연속 {MIN_QUARTERS}분기를 만족하는 티커가 하나도 없습니다.\n"
        f"  - DB의 분기 수가 부족하거나(--quarters 60 재수집 필요)\n"
        f"  - QUARTER_TOL_DAYS({QUARTER_TOL_DAYS}) 또는 MIN_QUARTERS({MIN_QUARTERS}) 설정을 확인하세요."
    )

revenue_filtered = pd.concat(kept_rows, ignore_index=True).copy()
revenue_filtered["cycle_group"] = revenue_filtered["date"].dt.month % 3
group_labels = {0: "표준그룹(3,6,9,12월분기말)", 1: "그룹B(4,7,10,1월분기말)", 2: "그룹C(5,8,11,2월분기말)"}
# 참고: 위 매핑은 '분기말 월'로 그룹을 나눈 것 -> 실제 분기 구성 월은 (분기말-2, 분기말-1, 분기말)

# --- cycle_group 단일화 -------------------------------------------------
_straddle = revenue_filtered.groupby("ticker")["cycle_group"].nunique()
_straddle = _straddle[_straddle > 1]
if len(_straddle):
    print(f"[경고] cycle_group이 2개 이상인 티커: {len(_straddle):,}개 "
          f"(예: {', '.join(_straddle.index[:8])})")
    if FORCE_SINGLE_CYCLE_GROUP:
        _dominant = (revenue_filtered.groupby("ticker")["cycle_group"]
                     .agg(lambda s: s.mode().iat[0]))
        _n0 = len(revenue_filtered)
        revenue_filtered = revenue_filtered[
            revenue_filtered["cycle_group"] == revenue_filtered["ticker"].map(_dominant)
        ].copy()
        print(f"        → 최빈 그룹으로 통일, 소수 그룹 행 {_n0 - len(revenue_filtered):,}건 제외")
else:
    print("[확인] 모든 티커가 단일 cycle_group에 속합니다.")

revenue_yoy_by_group = {}
for g_id, label in group_labels.items():
    df_g = revenue_filtered[revenue_filtered["cycle_group"] == g_id]
    pivot = df_g.pivot_table(index="date", columns="ticker", values="revenue", aggfunc="last")
    yoy_rows = []
    for ticker in pivot.columns:
        s = pivot[ticker].dropna()
        yoy = compute_revenue_yoy(s)
        if not yoy.empty:
            tmp = yoy.reset_index(); tmp.columns = ["date", "revenue_yoy"]; tmp["ticker"] = ticker
            yoy_rows.append(tmp)
    revenue_yoy_by_group[label] = (
        pd.concat(yoy_rows, ignore_index=True)[["ticker", "date", "revenue_yoy"]]
        if yoy_rows else pd.DataFrame(columns=["ticker", "date", "revenue_yoy"])
    )
    print(f"  - {label}: 티커 {revenue_yoy_by_group[label]['ticker'].nunique():,}개, "
          f"{len(revenue_yoy_by_group[label]):,}건")


[매출] 원본: 티커 1,974개 / 레코드 111,755건 (티커당 평균 56.6분기, 중복 제거 3건)
[매출] 연속 20분기 이상 만족 티커: 1,763개
[경고] cycle_group이 2개 이상인 티커: 30개 (예: AMCR, ATRI, BBY, BERY, CPE, CSR, CWST, DEI)
        → 최빈 그룹으로 통일, 소수 그룹 행 121건 제외
  - 표준그룹(3,6,9,12월분기말): 티커 1,578개, 83,573건
  - 그룹B(4,7,10,1월분기말): 티커 128개, 6,812건
  - 그룹C(5,8,11,2월분기말): 티커 57개, 3,172건


In [4]:

# 회계 캘린더 스냅 점검 (cycle_group이 티커당 값 1개면 정상)
CHECK_TICKERS = ["AAPL", "AMAT", "NVDA", "WMT", "ADI"]

_chk = revenue_filtered[revenue_filtered["ticker"].isin(CHECK_TICKERS)]
print(_chk.groupby("ticker").agg(
    n_quarters=("date", "count"),
    first_date=("date", "min"),
    last_date=("date", "max"),
    cycle_groups=("cycle_group", lambda s: sorted(s.unique())),
))


        n_quarters first_date  last_date cycle_groups
ticker                                               
AAPL            60 2011-06-30 2026-03-31          [0]
ADI             38 2017-01-31 2026-04-30          [1]
AMAT            60 2011-07-31 2026-04-30          [1]
NVDA            60 2011-07-31 2026-04-30          [1]
WMT             60 2011-07-31 2026-04-30          [1]


## ② 수출 HS코드 로드 + 연속 60개월 필터 + 분기합산 + YoY

In [5]:

MIN_MONTHS = 60
MONTH_TOL_DAYS = 35

trade_long = load_trade_monthly_long(engine, direction=DIRECTION)
n_total_hs = trade_long["hs_code"].nunique()

# HS코드당 월 수 분포 — self-join이 데이터를 날렸는지 즉시 드러난다
_months_per_hs = trade_long.groupby("hs_code")["date"].nunique()
print(f"[{DIRECTION}] HS코드당 월 수: 중앙값 {_months_per_hs.median():.0f} / "
      f"최소 {_months_per_hs.min()} / 최대 {_months_per_hs.max()}")
if _months_per_hs.median() < MIN_MONTHS:
    print(f"  ⚠️  중앙값이 {MIN_MONTHS}개월 미만입니다. "
          f"수출 테이블의 최신버전 self-join이 데이터를 과도하게 걸러내고 있을 수 있습니다.")

kept, n_qualified_hs = [], 0
for hs_code, g in trade_long.groupby("hs_code"):
    run_dates = get_trailing_consecutive_run(g["date"].tolist(), tol_days=MONTH_TOL_DAYS)
    if len(run_dates) >= MIN_MONTHS:
        n_qualified_hs += 1
        kept.append(g[g["date"].isin(run_dates)])

print(f"[{DIRECTION}] 연속 {MIN_MONTHS}개월 이상 만족 HS코드: {n_qualified_hs:,}개 (전체 {n_total_hs:,}개 중)")

if not kept:
    raise RuntimeError(
        f"연속 {MIN_MONTHS}개월을 만족하는 HS코드가 없습니다.\n"
        f"  위의 'HS코드당 월 수' 출력을 확인하세요. 중앙값이 1에 가까우면 self-join 문제입니다."
    )

trade_filtered = pd.concat(kept, ignore_index=True)
monthly_panel = trade_filtered.pivot_table(index="date", columns="hs_code", values="value", aggfunc="sum")

# 핵심: rolling(3).sum() 한 번으로 3가지 분기 패턴(표준/그룹B/그룹C) 전부의 분기합계를 커버
trailing_quarter_panel = monthly_panel.rolling(3, min_periods=3).sum()
trade_yoy_panel = trailing_quarter_panel.pct_change(12) * 100.0

print(f"[{DIRECTION}] HS코드 YoY 패널 shape: {trade_yoy_panel.shape}")
print(f"[{DIRECTION}] 패널 기간: {trade_yoy_panel.index.min():%Y-%m} ~ {trade_yoy_panel.index.max():%Y-%m}")


[export] self-join 적용 전 행 수: 613,135
[export] 로드 완료: 75,564행 / HS코드 500개
[export] HS코드당 월 수: 중앙값 161 / 최소 52 / 최대 161
[export] 연속 60개월 이상 만족 HS코드: 469개 (전체 500개 중)
[export] HS코드 YoY 패널 shape: (161, 469)
[export] 패널 기간: 2013-01 ~ 2026-05


## ③ 그룹별 상관계수 계산 (벡터화)

In [6]:

# n=4에서 r=0.66 같은 값은 통계적으로 무의미하다(p=0.34). 12분기(3년) 이상을 기본으로 둔다.
MIN_CORR_PERIODS = 12

try:
    from scipy import stats as _sps
    _HAS_SCIPY = True
except ImportError:
    _HAS_SCIPY = False
    print("[안내] scipy 미설치 — p_value 컬럼을 생략합니다.")

all_results = []
_miss_dates_total, _all_dates_total = 0, 0

for group_label, rev_yoy_df in revenue_yoy_by_group.items():
    if rev_yoy_df.empty:
        continue
    rev_pivot = rev_yoy_df.pivot_table(index="date", columns="ticker", values="revenue_yoy", aggfunc="last")

    for ticker in rev_pivot.columns:
        rev_series = rev_pivot[ticker].dropna()

        # 매출 리포트 날짜가 무역 패널 인덱스에 실제로 있는지 확인 (조용한 NaN 방지)
        _all_dates_total += len(rev_series)
        _miss_dates_total += (~rev_series.index.isin(trade_yoy_panel.index)).sum()

        trade_aligned = trade_yoy_panel.reindex(rev_series.index)

        corr_vec = trade_aligned.corrwith(rev_series)
        n_obs = (trade_aligned.notna() & rev_series.notna().values[:, None]).sum(axis=0)

        keep_idx = n_obs[n_obs >= MIN_CORR_PERIODS].index
        if len(keep_idx) == 0:
            continue

        out = pd.DataFrame({
            "ticker": ticker, "hs_code": keep_idx,
            "correlation": corr_vec.loc[keep_idx].values,
            "n_periods": n_obs.loc[keep_idx].values,
            "group": group_label,
        })
        all_results.append(out)

correlation_df = pd.concat(all_results, ignore_index=True) if all_results else pd.DataFrame()
correlation_df = correlation_df.dropna(subset=["correlation"])

if _all_dates_total:
    print(f"[정렬] 매출 리포트 날짜 {_all_dates_total:,}개 중 "
          f"무역 패널에 없는 날짜 {_miss_dates_total:,}개 "
          f"({_miss_dates_total / _all_dates_total * 100:.1f}%)")

if len(correlation_df):
    if _HAS_SCIPY:
        _r = correlation_df["correlation"].to_numpy(dtype=float)
        _n = correlation_df["n_periods"].to_numpy(dtype=float)
        with np.errstate(invalid="ignore", divide="ignore"):
            _t = _r * np.sqrt((_n - 2) / np.clip(1 - _r ** 2, 1e-12, None))
        correlation_df["p_value"] = 2 * _sps.t.sf(np.abs(_t), df=np.clip(_n - 2, 1, None))

    print(f"전체 상관계수 레코드: {len(correlation_df):,}건 "
          f"(티커 {correlation_df['ticker'].nunique():,}개 x "
          f"HS코드 {correlation_df['hs_code'].nunique():,}개)")
    print(f"  |r| >= 0.5 : {(correlation_df['correlation'].abs() >= 0.5).sum():,}건")
    print(f"  |r| >= 0.7 : {(correlation_df['correlation'].abs() >= 0.7).sum():,}건")
    print("  ※ 수십만 쌍을 동시에 스캔하므로 다중비교 문제가 큽니다. "
          "상위 상관계수는 검증된 관계가 아니라 가설 후보로만 다루세요.")
else:
    print("상관계수 레코드가 없습니다.")

save_path = r"C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_trade_revenue_corr"
os.makedirs(save_path, exist_ok=True)
fp = os.path.join(save_path, f"revenue_{DIRECTION}_corr_readonly_{pd.Timestamp.today():%Y%m%d}.csv")
correlation_df.to_csv(fp, index=False, encoding="utf-8-sig")
print(f"[저장] {fp}")


[정렬] 매출 리포트 날짜 93,557개 중 무역 패널에 없는 날짜 5,066개 (5.4%)
전체 상관계수 레코드: 826,636건 (티커 1,763개 x HS코드 469개)
  |r| >= 0.5 : 83,484건
  |r| >= 0.7 : 14,209건
  ※ 수십만 쌍을 동시에 스캔하므로 다중비교 문제가 큽니다. 상위 상관계수는 검증된 관계가 아니라 가설 후보로만 다루세요.
[저장] C:\Users\82108\OneDrive\바탕 화면\investment\data\analysis_results\US_trade_revenue_corr\revenue_export_corr_readonly_20260727.csv


## ④ 조회 예시

In [15]:

TARGET_TICKER = "MU"

ticker_result = get_top_hscodes_for_ticker(correlation_df, ticker=TARGET_TICKER, n=15)
print(f"[{TARGET_TICKER}] 상관계수 상위 HS 코드")
if ticker_result.empty:
    print("  해당 티커의 결과가 없습니다. revenue_filtered에 들어갔는지 확인하세요.")
else:
    print(ticker_result.to_string(index=False))

print()
hs_result = get_top_tickers_for_hscode(correlation_df, hs_code="854232", n=15)
print("[854232] 상관계수 상위 ticker")
print(hs_result.to_string(index=False))


[MU] 상관계수 상위 HS 코드
ticker hs_code  correlation  n_periods             group      p_value
    MU  854232     0.720807         49 그룹C(5,8,11,2월분기말) 5.204566e-09
    MU  853400     0.675675         49 그룹C(5,8,11,2월분기말) 1.000818e-07
    MU  847330     0.664322         49 그룹C(5,8,11,2월분기말) 1.944819e-07
    MU  847150     0.655916         49 그룹C(5,8,11,2월분기말) 3.123775e-07
    MU  760200     0.585267         49 그룹C(5,8,11,2월분기말) 1.001248e-05
    MU  740400     0.545245         49 그룹C(5,8,11,2월분기말) 5.096176e-05
    MU  853670     0.531507         49 그룹C(5,8,11,2월분기말) 8.504895e-05
    MU  711291     0.530060         49 그룹C(5,8,11,2월분기말) 8.964646e-05
    MU  711299     0.529892         49 그룹C(5,8,11,2월분기말) 9.019538e-05
    MU  711011     0.527840         49 그룹C(5,8,11,2월분기말) 9.714390e-05
    MU  852351     0.506182         49 그룹C(5,8,11,2월분기말) 2.067315e-04
    MU  841989     0.503765         49 그룹C(5,8,11,2월분기말) 2.242106e-04
    MU  440399     0.494643         49 그룹C(5,8,11,2월분기말) 3.029488e-04
 

## ⑤ 진단 (문제 재발 시)

In [8]:

# 특정 티커가 결과에서 빠졌을 때 원인을 좁히는 진단
DIAG_TICKERS = ("AAPL", "AMAT")

chk = pd.read_sql(text(f"""
    SELECT ticker, period, date, value
    FROM {TABLE_REVENUE}
    WHERE ticker IN ({','.join(repr(t) for t in DIAG_TICKERS)})
      AND item = 'revenue'
      AND period IN ('Q1','Q2','Q3','Q4')
      AND value IS NOT NULL AND value > 0
"""), engine)
chk["date"] = pd.to_datetime(chk["date"])

print("중복 점검 (count == nunique 여야 정상):")
print(chk.groupby("ticker")["date"].agg(["count", "nunique"]), "\n")

for tk, g in chk.groupby("ticker"):
    raw = sorted(g["date"].unique())
    print("=" * 64)
    print(f"{tk}  원본 분기말 {len(raw)}개  (최근: {pd.Timestamp(raw[-1]).date()})")
    for label, fn in [("calendar(구버전)", _month_end_calendar), ("fiscal(현재)", _month_end_fiscal)]:
        snapped = sorted({fn(d) for d in raw})
        run = get_trailing_consecutive_run(snapped, tol_days=QUARTER_TOL_DAYS)
        gaps = [(snapped[i-1].date(), snapped[i].date(), (snapped[i] - snapped[i-1]).days)
                for i in range(1, len(snapped))
                if (snapped[i] - snapped[i-1]).days > QUARTER_TOL_DAYS]
        print(f"  [{label:16s}] run={len(run):3d}분기 "
              f"{'통과' if len(run) >= MIN_QUARTERS else '탈락'} "
              f"/ cycle_group={sorted({d.month % 3 for d in run})}")
        if gaps:
            print(f"      {QUARTER_TOL_DAYS}일 초과 구간: {gaps}")


중복 점검 (count == nunique 여야 정상):
        count  nunique
ticker                
AAPL       60       60
AMAT       60       60 

AAPL  원본 분기말 60개  (최근: 2026-03-28)
  [calendar(구버전)   ] run= 13분기 탈락 / cycle_group=[0, 1]
      100일 초과 구간: [(datetime.date(2016, 12, 31), datetime.date(2017, 4, 30), 120), (datetime.date(2022, 12, 31), datetime.date(2023, 4, 30), 120)]
  [fiscal(현재)      ] run= 60분기 통과 / cycle_group=[0]
AMAT  원본 분기말 60개  (최근: 2026-04-26)
  [calendar(구버전)   ] run= 17분기 탈락 / cycle_group=[1, 2]
      100일 초과 구간: [(datetime.date(2016, 1, 31), datetime.date(2016, 5, 31), 121), (datetime.date(2021, 1, 31), datetime.date(2021, 5, 31), 120), (datetime.date(2022, 1, 31), datetime.date(2022, 5, 31), 120)]
  [fiscal(현재)      ] run= 60분기 통과 / cycle_group=[1]


In [16]:
hs = "854232"
rank_tbl = (correlation_df.query("hs_code == @hs")
            .sort_values("correlation", ascending=False)
            .reset_index(drop=True))
pos = rank_tbl.index[rank_tbl.ticker == "MU"][0] + 1
print(f"854232에 대한 MU 순위: {pos} / {len(rank_tbl)}  (r={rank_tbl.loc[pos-1,'correlation']:.3f})")
print(f"전체 티커 r 중앙값: {rank_tbl.correlation.median():.3f}, "
      f"90%분위: {rank_tbl.correlation.quantile(0.9):.3f}")
print(rank_tbl.head(10).to_string(index=False))


854232에 대한 MU 순위: 3 / 1763  (r=0.721)
전체 티커 r 중앙값: -0.005, 90%분위: 0.278
ticker hs_code  correlation  n_periods              group      p_value
   ABR  854232     0.722781         20 표준그룹(3,6,9,12월분기말) 3.181399e-04
   APH  854232     0.722365         49 표준그룹(3,6,9,12월분기말) 4.651859e-09
    MU  854232     0.720807         49  그룹C(5,8,11,2월분기말) 5.204566e-09
   DCP  854232     0.663537         38 표준그룹(3,6,9,12월분기말) 5.608449e-06
   QXO  854232     0.635366         49 표준그룹(3,6,9,12월분기말) 9.373365e-07
   CDE  854232     0.633728         49 표준그룹(3,6,9,12월분기말) 1.019585e-06
  AVAV  854232     0.615919         49  그룹B(4,7,10,1월분기말) 2.468107e-06
  CTRE  854232     0.603232         49 표준그룹(3,6,9,12월분기말) 4.484991e-06
  CVLT  854232     0.585660         49 표준그룹(3,6,9,12월분기말) 9.843394e-06
   GSS  854232     0.582631         31 표준그룹(3,6,9,12월분기말) 5.835271e-04


In [17]:
import numpy as np

rev = (revenue_yoy_by_group["그룹C(5,8,11,2월분기말)"]
       .query("ticker=='MU'").set_index("date")["revenue_yoy"])
panel = trade_yoy_panel.reindex(rev.index)

common = panel.mean(axis=1)          # 469개 HS 평균 = 무역 전반 경기요인

def partial(y, x, z):
    m = y.notna() & x.notna() & z.notna()
    y, x, z = y[m], x[m], z[m]
    ry = y - np.polyval(np.polyfit(z, y, 1), z)   # 공통요인 제거
    rx = x - np.polyval(np.polyfit(z, x, 1), z)
    return np.corrcoef(ry, rx)[0, 1], m.sum()

for hs in ["854232", "852351", "853400", "760200", "711011"]:
    raw = rev.corr(panel[hs])
    pc, n = partial(rev, panel[hs], common)
    print(f"{hs}  단순 r={raw:+.3f}  →  경기요인 제거 후 r={pc:+.3f}  (n={n})")

854232  단순 r=+0.721  →  경기요인 제거 후 r=+0.724  (n=49)
852351  단순 r=+0.506  →  경기요인 제거 후 r=+0.516  (n=49)
853400  단순 r=+0.676  →  경기요인 제거 후 r=+0.682  (n=49)
760200  단순 r=+0.585  →  경기요인 제거 후 r=+0.583  (n=49)
711011  단순 r=+0.528  →  경기요인 제거 후 r=+0.535  (n=49)


In [18]:
hypotheses = {
    "MU":   "854232",   # 메모리 IC          (통과: 3/1763)
    "APH":  "854232",   # 커넥터             (통과: 2/1763)
    "AMAT": "848620",   # 반도체 제조장비
    "BA":   "880240",   # 항공기 15톤 초과
    "DE":   "843351",   # 콤바인 수확기
    "CAT":  "842952",   # 굴착기
    "LRCX": "848620",
}
for tk, hs in hypotheses.items():
    t = (correlation_df.query("hs_code == @hs and n_periods >= 40")
         .sort_values("correlation", ascending=False).reset_index(drop=True))
    hit = t.index[t.ticker == tk]
    if len(hit):
        i = hit[0]
        print(f"{tk:5s} x {hs}  순위 {i+1:5d}/{len(t):5d}  r={t.loc[i,'correlation']:+.3f}")
    else:
        print(f"{tk:5s} x {hs}  해당 없음")

MU    x 854232  순위     2/ 1593  r=+0.721
APH   x 854232  순위     1/ 1593  r=+0.722
AMAT  x 848620  순위     1/ 1593  r=+0.760
BA    x 880240  해당 없음
DE    x 843351  순위     1/ 1593  r=+0.701
CAT   x 842952  순위   104/ 1593  r=+0.638
LRCX  x 848620  순위     2/ 1593  r=+0.751


In [19]:
import numpy as np

def lead_lag(ticker, hs, group, max_shift=6):
    rev = (revenue_yoy_by_group[group].query("ticker == @ticker")
           .set_index("date")["revenue_yoy"])
    rows = []
    for k in range(-max_shift, max_shift + 1):
        tr = trade_yoy_panel[hs].shift(k).reindex(rev.index)   # k>0 = k개월 전 무역데이터
        m = rev.notna() & tr.notna()
        if m.sum() >= 20:
            rows.append((k, np.corrcoef(rev[m], tr[m])[0, 1], int(m.sum())))
    return rows

for tk, hs, g in [("MU","854232","그룹C(5,8,11,2월분기말)"),
                  ("APH","854232","표준그룹(3,6,9,12월분기말)"),
                  ("AMAT","848620","그룹B(4,7,10,1월분기말)"),
                  ("DE","843351","그룹B(4,7,10,1월분기말)")]:
    rows = lead_lag(tk, hs, g)
    best = max(rows, key=lambda x: x[1])
    line = "  ".join(f"{k:+d}:{r:+.2f}" for k, r, _ in rows)
    print(f"{tk:5s} x {hs}  최적 shift={best[0]:+d}개월 (r={best[1]:+.3f})")
    print(f"       {line}\n")

MU    x 854232  최적 shift=+1개월 (r=+0.772)
       -6:+0.39  -5:+0.45  -4:+0.50  -3:+0.62  -2:+0.69  -1:+0.71  +0:+0.72  +1:+0.77  +2:+0.75  +3:+0.70  +4:+0.65  +5:+0.56  +6:+0.54

APH   x 854232  최적 shift=+1개월 (r=+0.739)
       -6:+0.50  -5:+0.58  -4:+0.58  -3:+0.58  -2:+0.68  -1:+0.68  +0:+0.72  +1:+0.74  +2:+0.71  +3:+0.70  +4:+0.70  +5:+0.69  +6:+0.66

AMAT  x 848620  최적 shift=+0개월 (r=+0.760)
       -6:+0.34  -5:+0.43  -4:+0.54  -3:+0.67  -2:+0.72  -1:+0.76  +0:+0.76  +1:+0.72  +2:+0.72  +3:+0.67  +4:+0.56  +5:+0.53  +6:+0.48

DE    x 843351  최적 shift=+0개월 (r=+0.701)
       -6:+0.50  -5:+0.43  -4:+0.56  -3:+0.63  -2:+0.60  -1:+0.70  +0:+0.70  +1:+0.65  +2:+0.58  +3:+0.52  +4:+0.46  +5:+0.37  +6:+0.32



In [20]:
import numpy as np
import pandas as pd

def pit_backtest(ticker, hs, group, shift=1, train_min=16):
    """확장 윈도우 회귀. 각 시점에서 과거 데이터만 사용."""
    rev = (revenue_yoy_by_group[group].query("ticker == @ticker")
           .set_index("date")["revenue_yoy"]).sort_index()
    tr = trade_yoy_panel[hs].shift(shift).reindex(rev.index)
    m = rev.notna() & tr.notna()
    rev, tr = rev[m], tr[m]

    rows = []
    for i in range(train_min, len(rev)):
        y, x = rev.iloc[:i], tr.iloc[:i]
        b, a = np.polyfit(x, y, 1)
        rows.append({
            "date":   rev.index[i],
            "actual": rev.iloc[i],
            "model":  a + b * tr.iloc[i],
            "naive":  rev.iloc[i-4] if i >= 4 else y.mean(),  # 4분기 전 YoY
            "mean":   y.mean(),
        })
    df = pd.DataFrame(rows)
    out = {}
    for k in ["model", "naive", "mean"]:
        e = df[k] - df["actual"]
        out[k] = dict(MAE=np.abs(e).mean(), RMSE=np.sqrt((e**2).mean()))
    return df, out

for tk, hs, g, sh in [("MU","854232","그룹C(5,8,11,2월분기말)",1),
                      ("APH","854232","표준그룹(3,6,9,12월분기말)",1),
                      ("AMAT","848620","그룹B(4,7,10,1월분기말)",1),
                      ("DE","843351","그룹B(4,7,10,1월분기말)",1)]:
    _, s = pit_backtest(tk, hs, g, sh)
    imp = (1 - s["model"]["RMSE"] / s["naive"]["RMSE"]) * 100
    print(f"{tk:5s} RMSE  모델 {s['model']['RMSE']:6.2f} | "
          f"단순 {s['naive']['RMSE']:6.2f} | 평균 {s['mean']['RMSE']:6.2f}  "
          f"→ 개선 {imp:+.1f}%")

MU    RMSE  모델  53.02 | 단순  86.68 | 평균  76.18  → 개선 +38.8%
APH   RMSE  모델  12.92 | 단순  19.79 | 평균  18.98  → 개선 +34.7%
AMAT  RMSE  모델   9.85 | 단순  21.93 | 평균  15.97  → 개선 +55.1%
DE    RMSE  모델  15.19 | 단순  26.95 | 평균  20.07  → 개선 +43.6%


In [21]:
import numpy as np, pandas as pd

def pit_backtest_v2(ticker, hs, group, shift=1, train_min=16):
    rev = (revenue_yoy_by_group[group].query("ticker == @ticker")
           .set_index("date")["revenue_yoy"]).sort_index()
    tr = trade_yoy_panel[hs].shift(shift).reindex(rev.index)
    m = rev.notna() & tr.notna()
    rev, tr = rev[m], tr[m]

    rows = []
    for i in range(train_min, len(rev)):
        y  = rev.iloc[1:i].values          # 타깃
        L  = rev.iloc[0:i-1].values        # 직전 분기 YoY
        X  = tr.iloc[1:i].values           # 무역 YoY
        y_last, x_now = rev.iloc[i-1], tr.iloc[i]

        # 1) 무역 단독
        b, a = np.polyfit(X, y, 1)
        p_trade = a + b * x_now
        # 2) AR(1) 단독
        b2, a2 = np.polyfit(L, y, 1)
        p_ar = a2 + b2 * y_last
        # 3) 결합 (AR(1) + 무역)
        A = np.column_stack([np.ones(len(y)), L, X])
        c = np.linalg.lstsq(A, y, rcond=None)[0]
        p_both = c[0] + c[1]*y_last + c[2]*x_now

        rows.append(dict(date=rev.index[i], actual=rev.iloc[i],
                         trade=p_trade, ar1=p_ar, both=p_both))

    df = pd.DataFrame(rows)
    r = {k: np.sqrt(((df[k]-df["actual"])**2).mean()) for k in ["trade","ar1","both"]}
    return df, r

print(f"{'':6} {'무역단독':>9} {'AR(1)':>9} {'결합':>9} | {'결합 vs AR(1)':>14}")
print("-"*56)
for tk, hs, g in [("MU","854232","그룹C(5,8,11,2월분기말)"),
                  ("APH","854232","표준그룹(3,6,9,12월분기말)"),
                  ("AMAT","848620","그룹B(4,7,10,1월분기말)"),
                  ("DE","843351","그룹B(4,7,10,1월분기말)")]:
    _, r = pit_backtest_v2(tk, hs, g)
    gain = (1 - r["both"]/r["ar1"]) * 100
    print(f"{tk:6} {r['trade']:9.2f} {r['ar1']:9.2f} {r['both']:9.2f} | {gain:+13.1f}%")

            무역단독     AR(1)        결합 |    결합 vs AR(1)
--------------------------------------------------------
MU         52.88     43.16     39.54 |          +8.4%
APH        12.95      7.70      6.58 |         +14.5%
AMAT        9.29      8.53      7.36 |         +13.7%
DE         15.21     10.92     10.82 |          +0.9%
